# 02 · English TTS benchmark (Colab GPU)

Every model reads the same 20 sentences (10 everyday assistant replies + 10 stress sentences with
rupee amounts, dates, codes and Indian names). Audio is saved to `MyDrive/local-voice-research/tts/`
and speed and memory go to `results.csv`.

**Order matters.** TTS packages often pin their own library versions, so each model has its own
install cell. After each install: **Runtime → Restart session**, run **Setup**, then the model cell.
Scoring (section 6) runs last, in a **fresh** runtime.

30-minute rule: if a model will not install or run, screenshot the error and move on.

## 1 · Setup (run after every restart)

In [ ]:
import os, sys, subprocess, importlib
from google.colab import drive, userdata
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
ROOT = "/content/drive/MyDrive/local-voice-research"
RESULTS, TTS = f"{ROOT}/results", f"{ROOT}/tts"
os.makedirs(RESULTS, exist_ok=True); os.makedirs(TTS, exist_ok=True)
if os.path.isdir("/content/lvr"):
    subprocess.run(["git", "-C", "/content/lvr", "pull", "-q"])
else:
    subprocess.run(["git", "clone", "-q", "https://github.com/Kiranrossi/local-voice-research", "/content/lvr"])
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "whisper_normalizer"])
sys.path.insert(0, "/content/lvr/src")
import harness as H, tts_bench as T
importlib.reload(H); importlib.reload(T)
import gc, numpy as np, torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
print(len(T.SENTENCES), "sentences ready")

def free_gpu():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

## 2 · Kokoro-82M (Apache 2.0)

Voice `af_heart` (American English, the model's highest-rated preset).

In [ ]:
!apt-get -qq -y install espeak-ng > /dev/null
!pip install -q "kokoro>=0.9" soundfile

In [ ]:
from kokoro import KPipeline

class Kokoro:
    sr = 24000
    def __init__(self, voice="af_heart"): self.voice = voice
    def load(self):
        try:
            self.p = KPipeline(lang_code="a", device="cuda")
        except TypeError:
            self.p = KPipeline(lang_code="a")
    def synthesize(self, text):
        parts = [a for _, _, a in self.p(text, voice=self.voice)]
        audio = np.concatenate([p.cpu().numpy() if hasattr(p, "cpu") else np.asarray(p) for p in parts])
        return audio, self.sr

model = Kokoro()
T.bench(model, "kokoro-82m", TTS, RESULTS, variant="82M af_heart", quantization="fp32", runtime="kokoro")
del model; free_gpu()

## 3 · Qwen3-TTS 0.6B CustomVoice (Apache 2.0)

Uses a preset speaker. If the model id or speaker name is rejected, the error lists the valid options:
copy one into the cell, or screenshot it and move on.

In [ ]:
!pip install -q -U qwen-tts

In [ ]:
from qwen_tts import Qwen3TTSModel

class QwenTTS:
    def __init__(self, model_id, speaker="Ryan"): self.model_id, self.speaker = model_id, speaker
    def load(self):
        self.m = Qwen3TTSModel.from_pretrained(self.model_id, device_map="cuda:0", dtype=torch.float16)
    def synthesize(self, text):
        wavs, sr = self.m.generate_custom_voice(text=text, language="English", speaker=self.speaker)
        w = wavs[0] if isinstance(wavs, (list, tuple)) else wavs
        w = w.cpu().numpy() if hasattr(w, "cpu") else np.asarray(w)
        return w, sr

model = QwenTTS("Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice")
T.bench(model, "qwen3-tts-0.6b", TTS, RESULTS, variant="0.6B CustomVoice", runtime="qwen-tts")
del model; free_gpu()

## 4 · VoxCPM2 (Apache 2.0, 2B)

In [ ]:
!pip install -q -U voxcpm

In [ ]:
from voxcpm import VoxCPM

class VoxCPM2:
    def load(self):
        self.m = VoxCPM.from_pretrained("openbmb/VoxCPM2")
        self.sr = next((v for v in (getattr(self.m, "sample_rate", None),
                                    getattr(getattr(self.m, "tts_model", None), "sample_rate", None)) if v), None)
        print("sample rate:", self.sr)
        assert self.sr, "Could not find the sample rate: screenshot this and move on"
    def synthesize(self, text):
        w = self.m.generate(text=text)
        w = w.cpu().numpy() if hasattr(w, "cpu") else np.asarray(w)
        return w, self.sr

model = VoxCPM2()
T.bench(model, "voxcpm2", TTS, RESULTS, variant="2B", runtime="voxcpm")
del model; free_gpu()

## 5 · Chatterbox (MIT, 0.5B)

Installed last because it may replace PyTorch with its own pinned version.

In [ ]:
!pip install -q chatterbox-tts

In [ ]:
from chatterbox.tts import ChatterboxTTS

class Chatterbox:
    def load(self):
        self.m = ChatterboxTTS.from_pretrained(device="cuda")
    def synthesize(self, text):
        w = self.m.generate(text)
        return w.squeeze().cpu().numpy(), self.m.sr

model = Chatterbox()
T.bench(model, "chatterbox", TTS, RESULTS, variant="0.5B default voice", quantization="fp32", runtime="chatterbox-tts")
del model; free_gpu()

## 6 · Score all TTS outputs (fresh runtime)

1. **Runtime → Disconnect and delete runtime**, then reconnect (this gives clean, default libraries).
2. Run **Setup** (section 1), then the cell below.

Round-trip intelligibility: Whisper large-v3-turbo transcribes each generated clip, and we compute WER
against the input text. Predicted naturalness: UTMOS (1 to 5, higher is better).

In [ ]:
from transformers import pipeline
asr = pipeline("automatic-speech-recognition", model="openai/whisper-large-v3-turbo",
               dtype=torch.float16, device="cuda:0")
def stt(audio):
    return asr({"raw": audio.copy(), "sampling_rate": 16000}, generate_kwargs={"language": "en", "task": "transcribe"})["text"]

try:
    utmos = torch.hub.load("tarepan/SpeechMOS:v1.2.0", "utmos22_strong", trust_repo=True).cuda().eval()
    def mos(audio):
        with torch.no_grad():
            return utmos(torch.from_numpy(audio).unsqueeze(0).cuda(), 16000).item()
except Exception as e:
    print("UTMOS unavailable, scoring intelligibility only:", e); mos = None

rows = T.score(TTS, RESULTS, stt, mos)

import pandas as pd
speed = pd.read_csv(f"{RESULTS}/results.csv"); speed = speed[speed.task == "tts"]
print(speed[["model", "device_name", "rtf", "latency_median_s", "peak_mem_gb", "notes"]].to_string(index=False))

In [ ]:
# Download the generated audio and scores, then upload the zip to the chat
!cd /content/drive/MyDrive/local-voice-research && zip -qr /content/tts_results.zip tts results/tts_scored.csv results/results.csv && echo zipped
from google.colab import files; files.download("/content/tts_results.zip")